In [4]:
# 모델 이름

from lib.utils.model_list import display_model_list

display_model_list()

                             Model Name
0                   cspresnet50.ra_in1k
1               eca_resnet33ts.ra2_in1k
2                 ecaresnet26t.ra2_in1k
3                ecaresnet50d.miil_in1k
4         ecaresnet50d_pruned.miil_in1k
5                  ecaresnet50t.a1_in1k
6                  ecaresnet50t.a2_in1k
7                  ecaresnet50t.a3_in1k
8                 ecaresnet50t.ra2_in1k
9               ecaresnet101d.miil_in1k
10       ecaresnet101d_pruned.miil_in1k
11               ecaresnet269d.ra2_in1k
12             ecaresnetlight.miil_in1k
13                gcresnet33ts.ra2_in1k
14                 gcresnet50t.ra2_in1k
15  inception_resnet_v2.tf_ens_adv_in1k
16          inception_resnet_v2.tf_in1k
17       lambda_resnet26rpt_256.c1_in1k
18             lambda_resnet26t.c1_in1k
19           lambda_resnet50ts.a1h_in1k
                             Model Name
0   efficientnet_b0.ra4_e3600_r224_in1k
1               efficientnet_b0.ra_in1k
2               efficientnet_b1.ft_in1k


In [ ]:
import pandas as pd
import pytorch_lightning as pl
from torch.utils.data.dataloader import DataLoader
from torchvision.transforms import Compose, Normalize, Resize, ToTensor

from dataset.slope_dataset import SlopeDataset
from lib.utils.path import output_path
from models.wheel_safe_classifier import WheelSafeClassifier

# 이미지 크기는 모델에 맞게 조절 (EfficientNet 기본 224, 240 등)
data_transform = Compose(
    [
        Resize((224, 224)),
        ToTensor(),
        Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ]
)

# CSV 또는 DataFrame 로드 (올려주신 이미지 기준 예시)
df = pd.read_csv(output_path() / 'slope_labels_007.csv')

train_dataset = SlopeDataset(df, transform=data_transform)
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)

# --- 모델 생성 및 학습 ---
model_system = WheelSafeClassifier(model_name='efficientnet_b0')

trainer = pl.Trainer(
    max_epochs=50,
    accelerator='gpu',
    devices=1,
    precision=16,  # 경사도 연산 가속
)

trainer.fit(model_system, train_loader)

In [ ]:
import matplotlib.pyplot as plt

from lib.utils.path import data_path
from run.predict import execute


def show_image(image, angle):
    img = plt.imread(image)
    plt.imshow(img)
    plt.title(f'Predicted Angle: {angle:.2f}°')
    plt.axis('off')
    plt.show()


execute('efficientnet_b0', data_path() / 'test', show_image)